# Welcome to the Local Particle Filter (LPF) tutorial


## Prerequisites

<u>This tutorial requires completion of the **0.qg_tutorial_start** tutorial</u>. We will use the truth run and initial ensemble generated from that tutorial for performing a set of experiments with LPF.

This tutorial also builds off of the steps introduced in **3.qg_LETKF_tutorial** and **6.qg_cycling_tutorial**.


# Introduction to particle filters
Particle filters (PFs) are sequential Monte Carlo techniques designed to handle data assimilation problems that feature non-Gaussian error distributions in either model forecasts or observations; see Doucet et al. (2001) for a review. 

From a geoscience perspective, PFs are incredibly appealing:
* **Dynamical Consistency:** They inherently maintain the dynamical balances of the system during the data assimilation update steps.
* **Flexibility:** They naturally accommodate nonlinear measurement operators without requiring special mathematical treatment or linearizations.

### The Bootstrap Particle Filter

To understand how PFs work, it helps to start with the bootstrap particle filter. Instead of assuming a Gaussian distribution, PFs represent the probability density function of the system state using an ensemble of $N$ discrete samples, or "particles". 

Let $x_i$ represent the state of the system at time $i$, and $y_{1:i-1}$ represent all previous observations. If we have an ensemble of $N$ equally weighted forecast particles, denoted as $x_i^{f,(n)}$ for $n = 1, \dots, N$, the **prior distribution** is approximated empirically using a sum of Dirac delta functions:

$$p(x_i | y_{1:i-1}) \approx \sum_{n=1}^N \frac{1}{N} \delta(x_i - x_i^{(n)})$$

When a new observation $y_i$ becomes available, Bayes' rule tells us that the posterior distribution is proportional to the prior multiplied by the likelihood of particles, given the new observation. 

In the bootstrap filter, we evaluate the likelihood $p(y_i | x_i^{(n)})$ for each particle. This likelihood gives each particle a new unnormalized weight, $w_i^{(n)}$, based on how closely it matches the observation:

$$w_i^{(n)} \propto p(y_i | x_i^{(n)})$$

After normalizing these weights so they sum to one (let's call the normalized weight $\tilde{w}_i^{(n)}$), we can express the **posterior distribution** as:

$$p(x_i | y_{1:i}) \approx \sum_{n=1}^N \tilde{w}_i^{(n)} \delta(x_i - x_i^{(n)})$$

Typically, this is followed by a "resampling" step, where particles with high weights are duplicated and particles with low weights are discarded, returning us to an ensemble of equally weighted particles for the next forecast cycle.

### The High-Dimensional Challenge: Why Localize?
Similar to EnKFs, traditional PFs struggle when applied to high-dimensional geophysical models, thus motivating the use of **Localized Particle Filters (LPFs)**.

Similar to EnKFS, localization takes a large data assimilation task and breaks it  into a collection of smaller, loosely connected problems. These localized regions can then be solved independently using relatively small ensembles.

### Lingering Challenges in the Real World
While localization is a necessary step toward applying PFs to high-dimensional systems, it is rarely sufficient to maintain enough diversity in weights for real-world geophysical tasks. 

As we will see in this tutorial, the well-documented vulnerability of PFs to **weight collapse**—where a single particle accumulates all the statistical weight, meaning $\tilde{w}_i^{(n)} \approx 1$ for one particle and $0$ for the rest—persists even within localized observation neighborhoods. As a result, filter degeneracy remains an unavoidable challenge in systems observed by dense clusters of highly accurate measurements or when the prior statistics are significantly biased.

To avoid weight collapse locally (i.e., outside of regions where localization tapers the influence of observations to zero), the LPF used in this exercise applies a "tempering" to weights, meaning that it raises pre-normalized weights to a power less than 1, then assimilates observations multiple times until these powers sum to 1. This step provides an additional guardrail for maintaining variance in the ensemble, similar to variance inflation in EnKFs. The tempering is controlled by enforcing a minimum "effective ensemble size" ($N_{eff}$) for particles:

$$N_{eff} = \frac{1}{\sum_{n=1}^N (\tilde{w}_i^{(n)})^2},$$

where the threshold $N_{eff}$ is set as a fraction of the unweighted ensemble size, $N$.

More specifics on how the LPF implements each of these steps and updates particles to reflect a localized posterior density are described in [Poterjoy (2022)](https://rmets.onlinelibrary.wiley.com/doi/full/10.1002/qj.4328).

### Set Environment Variables
We now set environment variables and verify that the required executables exist. **You need to do this anytime you open this notebook for a new localhost session**

Unlike previous tutorials, this one requires the latest version of OOPS, which exists in directory build-v2.

In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU
export JEDI_BIN_v2=/home/nonroot/build-v2/bin

In [ ]:
# Verify LPF executables exist
ls $JEDI_BIN_v2/qg_gen_ens_pert_B.x
ls $JEDI_BIN_v2/qg_etkf.x
ls $JEDI_BIN_v2/qg_ens_mean_variance.x


### Table of LPF Experiments

| Experiment Name | LPF yaml file | Description |
| --- | --- | --- |
| exp_default | lpf.yaml | Default single-observation LPF experiment (compared to LETKF)
| exp_dbl_loc | lpf_dbl_loc.yaml | Double the localization scale
| exp_lrgNeff | lpf_lrgNeff.yaml | Increases the effective ensemble size while keeping iterations to 1
| exp_mult_obs  | lpf_mult_obs.yaml | Assimilates 100 observations
| exp_lpf  | lpf_template.yaml | Run 6 cycles of data assimilation
| exp_lpf (BONUS)  | lpf_template.yaml | Repeat previous experiment with twice the ensemble size

# Experiment 1: LPF with one observation
***

### Step 1 & 2: Verify that truth and background ensemble exist

Double-check the  files exist from the qgstart tutorial. If not, please run that tutorial first!

In [ ]:
ls $JEDI_EDU/qgstart/output/truth

In [ ]:
# Verify that ensemble files were produced
ls -S $JEDI_EDU/qgstart/output/bg/bkgd.ens.*P1D.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Background files don't exist! Please see the 0.qg_tutorial_start tutorial to generate the background ensemble"
fi

### Step 3: Verify observation file

In [ ]:
# Verify that ensemble files were produced
ls -S $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc
if [ $? -ne 0 ]; then
     echo "ERROR! Observation file doesn't exist! Please see the 0.qg_tutorial_start tutorial to generate the single observation"
fi

To verify the observation is correct, run the following python script and cat command to view the observation location and values. Note the $lon$, $lat$ should be $[150]$ and $[50]$, respectively, close to the northeastern part of our plot domain, the same as the dot shown in the above variance figure. 

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc \
        --output $JEDI_EDU/qgLPF/output/exp_default/plots/qg_oneob

In [ ]:
cat $JEDI_EDU/qgLPF/output/exp_default/plots/qg_oneob.txt

We can now run the LPF!

### Step 4: Run the LPF!

```yaml

time window:
  begin: &date_bgn 2009-12-30T18:00:00Z
  length: PT12H

local ensemble DA: 
  solver: PF  # Controls which flavor of ensemble solver is used
  inflation:    # Covariance inflation (applied after LETKF update)
    mult: 1.0   # Multiplicative inflation factor
  pf frac neff: 0.1 # This parameter enforces weights to have an effective ensemble size of 0.1 x N = 2
  pf max iter: 1 # Perform a single pass through observations while enforcing an effective ensemble size of 12

obs space:
  obsdatain:
    obsfile: /home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc
  obsdataout:
    obsfile: qgLPF/output/exp_default/obs/lpf_oneob.obs3d.nc
  obs type: Stream

In [ ]:
cd $JEDI_EDU
$JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf.yaml 

In [ ]:
ls $JEDI_EDU/qgLPF/output/exp_default/da

**Recall from past tutorials: The posterior analysis ensemble mean is the "000000" ensemble member.** 

We will now plot the results using the python plotting script. Let's examine the mean increments. That is, the mean ensemble analysis minus the mean ensemble background. (Alternatively, you could plot the contents of the `da/lpf_meaninc.an.2009-12-31T00:00:00Z.nc` file directly)

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLPF/output/exp_default/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgLPF/output/exp_default/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU/qgLPF/output/exp_default/plots/lpf_mean_increments --title "ens. mean anl inc."

#### Ensemble Mean Increments 

We first compare posterior mean increments to the LETKF. Note that these increments are structurally similar, as you would expect for the case where the prior distribution is Gaussian. Note that the LPF exhibits a smaller amplitude in the adjustment to the posterior mean. This difference is reflective of the small ensemble size and the fact that the posterior mean computed by the LPF is bounded by the particles. 

We should also keep in mind that the LETKF and LPF implement localization differently, which further complicates the comparison.

| LETKF | LPF |
|---|---|
| ![](./images/ensmean_inc_default_x_v2.jpg) | ![](images/lpf_mean_increments_x_diff.jpg) |




# Experiment 2: Change the horizontal localization length scale

***

### Steps 1-3: Truth, Prior Ensemble, Observations

This step can be skipped, as everything remains the same. 

### Step 4: Run the data assimilation:
Let's compare `lpf_dbl_loc.yaml` to the default experiment `lpf.yaml`:

In [ ]:
diff $JEDI_EDU/qgLPF/yamls/lpf.yaml $JEDI_EDU/qgLPF/yamls/lpf_dbl_loc.yaml

The main parameter that changes the length scale for localization is `lengthscale`, which is doubled to 8.0e6 (i.e. 8000 km). There are also changes to the output file locations for the analysis that replaced  `exp_default` by `exp_dbl_loc`, ensuring we do not overwrite any previous results.

You may run the LPF:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_dbl_loc.yaml

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLPF/output/exp_dbl_loc/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgLPF/output/exp_dbl_loc/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU/qgLPF/output/exp_dbl_loc/plots/lpf_mean_increments --title "ens. mean anl inc."

#### Ensemble Mean Increments - Doubled Loc
| LPF (original) | LPF (twice localization) |
|---|---|
| ![](images/lpf_mean_increments_x_diff.jpg) | ![](images/lpf_mean_increments_x_diff_dbl_loc.jpg) |


# Experiment 3: LPF with larger effective ensemble size

The LPF relies on an additional regularization of particle weights to maintain diversity in the ensemble when localization is insufficient to prevent weights from collapsing. Increasing the parameter that controls the minimum $N_{eff}$ will reduce updates to all posterior quantities.

### Steps 1-3: Truth, Background Ensemble, Observations

As in experiment 2, these files remain the same

### Step 4: Run LPF
Let us compare `lpf_90Neff.yaml` to the default experiment `lpf.yaml`. 

In [ ]:
diff $JEDI_EDU/qgLPF/yamls/lpf.yaml $JEDI_EDU/qgLPF/yamls/lpf_90Neff.yaml

Comparing these two yaml files, note that the change in parameter `pf frac neff` results in an increase in $N_{eff}$ to a value of 18, which forces weights to be closer to equal.

In [ ]:
cd $JEDI_EDU
$JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_90Neff.yaml 

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLPF/output/exp_90Neff/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgLPF/output/exp_90Neff/da/lpf_meanprior.an.2009-12-31T00:00:00Z.nc \
        --plotObsValues $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc  \
        --plotwind --fieldmax 50000000.0 \
        --output $JEDI_EDU/qgLPF/output/exp_90Neff/plots/lpf_mean_increments --title "ens. mean anl inc."

#### Ensemble Mean Increments - LPF
| LPF Neff = 2 | LPF Neff = 18 |
|---|---|
| ![](images/lpf_mean_increments_x_diff.jpg) | ![]( images/lpf_mean_increments_x_diff_90Neff.jpg) |





# Experiment 4: Assimilate multiple observations
***

We now embark on a more realistic situation in which 100 observations are assimilated all at once. Likewise, we will compare our results with those from a similarly-configured LETKF.

### Steps 1-3: Truth, Background Ensemble, Observations

The truth and background ensemble remain unchanged. We will copy the 100-observations file produced from the **0.qg_tutorial_start** tutorial  to the **exp_mult_obs** folder 


In [ ]:
cp $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc $JEDI_EDU/qgLPF/output/exp_mult_obs/obs
ls $JEDI_EDU/qgLPF/output/exp_mult_obs/obs

### Step 4: Run the LPF

Let's compare the 100-obs settings for `lpf_mult_obs.yaml` to the single-observation settings in `lpf.yaml`

In [ ]:
diff $JEDI_EDU/qgLPF/yamls/lpf.yaml $JEDI_EDU/qgLPF/yamls/lpf_mult_obs.yaml

In [ ]:
cd $JEDI_EDU
$JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_mult_obs.yaml 

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgLPF/output/exp_mult_obs/da/lpf_meanprior.an.2009-12-31T00\:00\:00Z.nc \
        --plotObsLocations $JEDI_EDU/qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgLPF/output/exp_mult_obs/plots/lpf_meaninc --title "analysis increment ENSMEAN"


| LETKF | LPF |
|---|---|
| ![](./images/letkf_meaninc_x_100obs_v2.jpg) | ![](images/lpf_mean_increments_x_diff_mult_obs.jpg) |



In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgLPF/output/exp_mult_obs/da/lpf.000000.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsLocations $JEDI_EDU/qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgLPF/output/exp_mult_obs/plots/lpf_meananl_error --title "mean analysis error" 
python ./plot.py qg fields \
        $JEDI_EDU/qgLPF/output/exp_mult_obs/da/lpf_meanprior.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotObsLocations $JEDI_EDU/qgLPF/output/exp_mult_obs/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgLPF/output/exp_mult_obs/plots/lpf_meanbkg_error --title "mean background error" 

## Prior and Posterior Mean Stream function

### LETKF
| | |
|---|---|
| ![](./images/letkf_meanbkg_error_x_100obs_v2.jpg) | ![](./images/letkf_meananl_error_x_100obs_v2.jpg) |

### LPF
| | |
|---|---|
| ![](./images/lpf_meanbkg_error_x_diff_100obs.jpg) | ![](images/lpf_meananl_error_x_diff_mult_obs.jpg) |

For Gaussian applications, the LPF will provide qualitatively similar results to the LETKF, despite relying on a very different set of assumptions for the prior PDF. Nevertheless, the Delta function approximation will lead to greater sampling deficiency than methods that rely on error covariances alone to characterize uncertainty. 

# Experiment 5: LPF Cycling with multiple observations
***

Finally, we will run cycling data assimilation with the LPF using a setup that resembles the LETKF used in **6.qg_cycling_tutorial**. 

To save time, we will restrict this exercise to 6 cycles.

In [ ]:
export ncycles=6 # If you want more than 6, you need to extend the truth simulation in 0.qg_tutorial_start!!!

### Make Observations for All Cycles


For this cycling experiment, we will use 100 randomly generated observations for each cycle time, to mimic the sort of cycling performed with real observations in operational NWP centers.

In [ ]:
cd $JEDI_EDU

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   forecast_hour="P$((icyc+15))D"
   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, forecast time = $forecast_hour
   
cat << EOF > ./qgLPF/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: ${curdate}
  filename: /home/nonroot/shared/EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.${forecast_hour}.nc
time window: 
  begin: ${curdate_minus6h}
  length: PT12H
observations:
  observers:
  - obs operator:
      obs type: Stream                             # The observation type is the stream function
    obs space:
      obsdataout:
        obsfile: /home/nonroot/shared/EDU/qgstart/output/obs/truth_cyc${icyc}.obs3d.nc
      obs type: Stream
      generate:
        begin: PT6H
        nval: 1
        obs density: 100                           # Generate 100 observations at random locations
        obs error: 4.0e6                           # Observation error standard deviation, in m^2/s
        obs period: PT12H
make obs: true                                     # Generate the observations
obs perturbations: true                            # Add random  measurements to the observations
EOF

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

After running, you can verify that there were yaml files created. There should be as many files as there are cycles (set by `ncycles`).

In [ ]:
cd $JEDI_EDU
ls ./qgLPF/yamls/makeobs3d_mult_obs_cyc*yaml

We can now run the `qg_hofx` program as in the qgstart tutorial to create observations for each cycle. Note that the outputted cycle observations are saved in the `qgstart/output/obs` directory, as in the qgstart tutorial. For efficiency, the output of the JEDI program has been redirected to log files, with a simple echo statement to confirm each cycle has successfully run or encountered an error.

In [ ]:
cd $JEDI_EDU
for icyc in $(seq 1 $ncycles); do
    $JEDI_BIN_v2/qg_hofx3d.x ./qgLPF/yamls/makeobs3d_mult_obs_cyc${icyc}.yaml >& ./qgLPF/output/makeobs_cyc${icyc}.log
    if [ $? -ne 0 ]; then
        echo "ERROR! Something went wrong running for cycle $icyc. Check ./qgLPF/output/makeobs_cyc${icyc}.log"
    else
        echo "SUCCESS run for cycle $icyc"
    fi
done

If you encounter any issues, check the output log created. Otherwise, let's quickly verify observations were created for all cycles with an `ls` command:

In [ ]:
cd $JEDI_EDU
ls  ./qgstart/output/obs/truth_cyc*.obs3d.nc

### Construct LPF and forecast yamls

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   # for window begin:
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   # for bgfile:
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")

   if [ $icyc -eq 1 ]; then
      bgfile="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
   else
      bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.fc.${curdate_minus1day}.P1D.nc"
   fi

   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, bgfile = $bgfile
   
   sed -e "s|@DATE_BGN|${curdate_minus6h}|g;s|@DATE_ANL|${curdate}|g" \
       -e "s|@CYC|$icyc|g;s|@BACKGROUND_FILE|${bgfile}|g" \
       ./qgLPF/yamls/lpf_template.yaml > ./qgLPF/yamls/lpf_cyc${icyc}.yaml

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   for imem in $(seq 1 20); do
      sed -e "s|%MEMBER%|$(printf "%06d\n" "$imem")|g;s|%MEM%|$imem|g;s|@DATE_IC|${curdate}|g" \
             ./qgLPF/yamls/forecast_template.yaml > ./qgLPF/yamls/forecast_cyc${icyc}_mem${imem}.yaml
   done 

    echo "Cycle $icyc: curdate = $curdate, forecast_cyc${icyc}_mem*.yaml files created!"

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done
  

### Run the cycling!

Everything is now ready to go. As in the "cycling" tutorial, we will perform a series of update and prediction steps to assimilate all of the observations we just created. Each step uses the yaml files created above.

In [ ]:
cd $JEDI_EDU
for icyc in $(seq 1 $ncycles); do
   start_time=`date +%s` # Needed to calculate time it takes to complete each step
   
   # First run lpf
   $JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_cyc${icyc}.yaml >& ./qgLPF/output/lpf_cyc${icyc}.log
   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running lpf for cycle $icyc. Check ./qgLPF/output/lpf_cyc${icyc}.log"
      break 
   else
      end_time_lpf=`date +%s`
      echo "SUCCESS lpf run for cycle $icyc, took $(( end_time_lpf - start_time )) seconds"
   fi
   
   # Then run QG ensemble forecast by looping over each member 
   for imem in $(seq 1 20); do
      $JEDI_BIN_v2/qg_forecast.x ./qgLPF/yamls/forecast_cyc${icyc}_mem${imem}.yaml >& ./qgLPF/output/exp_lpf/forecast_cyc${icyc}_mem${imem}.log
      if [ $? -ne 0 ]; then
         echo "ERROR! Something went wrong running forecast for cycle $icyc, member $imem. Check ./qgLPF/output/exp_lpf/forecast_cyc${icyc}_mem${imem}.log"
         break 2
      else
         echo "SUCCESS forecast run for cycle $icyc, member $imem"
      fi
   done
   end_time_fcst=`date +%s`
   echo "SUCCESS ensemble forecast run for cycle $icyc, took $(( end_time_fcst - end_time_lpf )) seconds"

done

We can then check the qgLPF/output/exp_lpf/da directory to see that analysis files were created. We will just list the content from member 1, but you should confirm that 20 members of output were produced in the aforementioned directory.


In [ ]:
cd $JEDI_EDU
echo "Ensemble Member 1 Analyses (used as ICs for member 1 forecast to the next cycle):"
ls ./qgLPF/output/exp_lpf/da/lpf.000001.an*nc
echo -e "\nEnsemble Member 1 Forecasts (used as backgrounds for each cycle):"
ls ./qgLPF/output/exp_lpf/da/lpf.1.fc*P1D.nc
echo -e "\nEnsemble Mean Analyses:"
ls ./qgLPF/output/exp_lpf/da/lpf.000000.an*nc
echo -e "\nEnsemble Mean Prior and Increments:"
ls ./qgLPF/output/exp_lpf/da/lpf_mean*nc

### Compute RMSE and Ensemble Variance

We will use the same Python utilities provided earlier to compute the domain-average prior and posterior RMSE and ensemble spread each cycle.

In [ ]:
cd $JEDI_EDU
output_filename="/home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt"

# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing rmse for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_rmse_layers.py \
          $JEDI_EDU/qgLPF/output/exp_lpf/da/lpf_meanprior.an.${curdate}.nc \
          $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   # Run for analysis mean file
   python ./plots_scripts/compute_rmse_layers.py \
          $JEDI_EDU/qgLPF/output/exp_lpf/da/lpf.000000.an.${curdate}.nc \
          $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          --v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

In [ ]:
cd $JEDI_EDU/
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
 for anorbg in bg an; do
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")
   if [ "$anorbg" == "bg" ]; then
      zpad=0
      if [ $icyc -eq 1 ]; then
         bgfile_mem1="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.1.${curdate_minus1day}.P1D.nc"
         bgfile="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
      else
         bgfile_mem1="qgLPF/output/exp_lpf/da/lpf.1.fc.${curdate_minus1day}.P1D.nc"
         bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.fc.${curdate_minus1day}.P1D.nc"
      fi
   else
       zpad=6
       bgfile_mem1="qgLPF/output/exp_lpf/da/lpf.000000.an.${curdate}.nc"
       bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.an.${curdate}.nc"
   fi

   # Build namelist for ens variance JEDI program
   cat << EOF > ./qgLPF/yamls/ens_variance_cyc${icyc}.yaml
background:
  date: $curdate
  filename: $bgfile_mem1
  state variables: [x,q,u,v]
ensemble:
  members from template:
    template:
      date: $curdate
      filename: $bgfile
      state variables: [x,q,u,v]
    pattern: %mem%
    zero padding: $zpad
    nmembers: 20
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
variance output:
  datadir:  qgLPF/output/exp_lpf/da
  exp: ${anorbg}EnsVariance
  type: diag
  date: $curdate
EOF
   # Run the ens variance program
   $JEDI_BIN_v2/qg_ens_mean_variance.x ./qgLPF/yamls/ens_variance_cyc${icyc}.yaml \
                            >& ./qgLPF/output/exp_lpf/ens_var_cyc${icyc}_${anorbg}.log

   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running qg_ens_variance.x for cycle $icyc. Check ./qgLPF/output/exp_lpf/ens_var_cyc${icyc}_${anorbg}.log"
      break
   else
      echo "SUCCESS qg_ens_variance.x run for cycle $icyc $anorbg"
   fi
 done #anorbg loop
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done #icyc loop

ls -lhtr ./qgLPF/output/exp_lpf/da/*EnsVariance*nc

In [ ]:
cd $JEDI_EDU
output_filename="/home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt"
# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing average ensemble variance for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_ensvar_layers.py \
          $JEDI_EDU/qgLPF/output/exp_lpf/da/bgEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   python ./plots_scripts/compute_ensvar_layers.py \
          $JEDI_EDU/qgLPF/output/exp_lpf/da/anEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

In [ ]:
cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth_withvar.py -m 6e7 $JEDI_EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt \
                             $JEDI_EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt \
                             -o $JEDI_EDU/qgLPF/output/exp_lpf/plots/rmse_sawtooth_withvar-20mem.png

<center><img src="images/rmse_sawtooth_withvar_lpf-20mem.png" alt="sawtooth" width="600"/></center>

The LPF reduces mean errors each cycle, but the errors by the end of cycling are greater than those we found from the LETKF in **6.qg_cycling_tutorial**. This finding should not be surprising, given that we are only using 20 members.

# Experiment 5: Double the ensemble size

We can use the same steps from the first tutorial to generate 40 members. There is no need to re-generate observations, but we do need to create additional ensemble members for the first cycle.

To save time, we will use a yaml file that is already configured to generate 40 perturbations.

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_gen_ens_pert_B.x $JEDI_EDU/qgLPF/yamls/genenspert_B_ens40.yaml

In [ ]:
# Verify that ensemble files were produced
cd $JEDI_EDU/qgstart/output/bg
ls bkgd.ens*nc

### Modify LPF and forecast yamls

Instead of `lpf_template.yaml`, use the 40-member yaml file template that has been configured for the larger ensemble size, called `lpf_template-40mem.yaml`. 

We also need to generate 40 forecast yaml files.

In [ ]:
diff $JEDI_EDU/qgLPF/yamls/lpf_template.yaml $JEDI_EDU/qgLPF/yamls/lpf_template-40mem.yaml

We also need to generate a new set of analysis and forecast yaml files based on the updated template and the increase in the number of members.

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   # for window begin:
   curdate_minus6h=$(date -u -d "$curdate -6 hours" +"%Y-%m-%dT%H:%M:%SZ")
   # for bgfile:
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")

   if [ $icyc -eq 1 ]; then
      bgfile="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
   else
      bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.fc.${curdate_minus1day}.P1D.nc"
   fi

   echo Cycle $icyc: curdate = $curdate, window begin = $curdate_minus6h, bgfile = $bgfile
   
   sed -e "s|@DATE_BGN|${curdate_minus6h}|g;s|@DATE_ANL|${curdate}|g" \
       -e "s|@CYC|$icyc|g;s|@BACKGROUND_FILE|${bgfile}|g" \
       ./qgLPF/yamls/lpf_template-40mem.yaml > ./qgLPF/yamls/lpf_cyc${icyc}.yaml

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

In [ ]:
cd $JEDI_EDU
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   for imem in $(seq 1 40); do
      sed -e "s|%MEMBER%|$(printf "%06d\n" "$imem")|g;s|%MEM%|$imem|g;s|@DATE_IC|${curdate}|g" \
             ./qgLPF/yamls/forecast_template.yaml > ./qgLPF/yamls/forecast_cyc${icyc}_mem${imem}.yaml
   done 

    echo "Cycle $icyc: curdate = $curdate, forecast_cyc${icyc}_mem*.yaml files created!"

   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done
  

### Run 40-member cycling data assimilation

This step is the same as above, except we need to loop over 40 forecast integrations each cycle.

In [ ]:
cd $JEDI_EDU
for icyc in $(seq 1 $ncycles); do
   start_time=`date +%s` # Needed to calculate time it takes to complete each step
   
   # First run lpf
   $JEDI_BIN_v2/qg_etkf.x ./qgLPF/yamls/lpf_cyc${icyc}.yaml >& ./qgLPF/output/lpf_cyc${icyc}.log
   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running lpf for cycle $icyc. Check ./qgLPF/output/lpf_cyc${icyc}.log"
      break 
   else
      end_time_lpf=`date +%s`
      echo "SUCCESS lpf run for cycle $icyc, took $(( end_time_lpf - start_time )) seconds"
   fi
   
   # Then run QG ensemble forecast by looping over each member 
   for imem in $(seq 1 40); do
      $JEDI_BIN_v2/qg_forecast.x ./qgLPF/yamls/forecast_cyc${icyc}_mem${imem}.yaml >& ./qgLPF/output/exp_lpf/forecast_cyc${icyc}_mem${imem}.log
      if [ $? -ne 0 ]; then
         echo "ERROR! Something went wrong running forecast for cycle $icyc, member $imem. Check ./qgLPF/output/exp_lpf/forecast_cyc${icyc}_mem${imem}.log"
         break 2
      else
         echo "SUCCESS forecast run for cycle $icyc, member $imem"
      fi
   done
   end_time_fcst=`date +%s`
   echo "SUCCESS ensemble forecast run for cycle $icyc, took $(( end_time_fcst - end_time_lpf )) seconds"

done

### Compute RMSE and ensemble variance to generate sawtooth graphic

In [ ]:
cd $JEDI_EDU
output_filename="/home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt"

# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing rmse for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_rmse_layers.py \
          $JEDI_EDU/qgLPF/output/exp_lpf/da/lpf_meanprior.an.${curdate}.nc \
          $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   # Run for analysis mean file
   python ./plots_scripts/compute_rmse_layers.py \
          $JEDI_EDU/qgLPF/output/exp_lpf/da/lpf.000000.an.${curdate}.nc \
          $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P$((icyc+15))D.nc \
          --v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done


cd $JEDI_EDU/
firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
 for anorbg in bg an; do
   curdate_minus1day=$(date -u -d "$curdate -24 hours" +"%Y-%m-%dT%H:%M:%SZ")
   if [ "$anorbg" == "bg" ]; then
      zpad=0
      if [ $icyc -eq 1 ]; then
         bgfile_mem1="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.1.${curdate_minus1day}.P1D.nc"
         bgfile="/home/nonroot/shared/EDU/qgstart/output/bg/bkgd.ens.%mem%.${curdate_minus1day}.P1D.nc"
      else
         bgfile_mem1="qgLPF/output/exp_lpf/da/lpf.1.fc.${curdate_minus1day}.P1D.nc"
         bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.fc.${curdate_minus1day}.P1D.nc"
      fi
   else
       zpad=6
       bgfile_mem1="qgLPF/output/exp_lpf/da/lpf.000000.an.${curdate}.nc"
       bgfile="qgLPF/output/exp_lpf/da/lpf.%mem%.an.${curdate}.nc"
   fi

   # Build namelist for ens variance JEDI program
   cat << EOF > ./qgLPF/yamls/ens_variance_cyc${icyc}.yaml
background:
  date: $curdate
  filename: $bgfile_mem1
  state variables: [x,q,u,v]
ensemble:
  members from template:
    template:
      date: $curdate
      filename: $bgfile
      state variables: [x,q,u,v]
    pattern: %mem%
    zero padding: $zpad
    nmembers: 20
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
variance output:
  datadir:  qgLPF/output/exp_lpf/da
  exp: ${anorbg}EnsVariance
  type: diag
  date: $curdate
EOF
   # Run the ens variance program
   $JEDI_BIN_v2/qg_ens_mean_variance.x ./qgLPF/yamls/ens_variance_cyc${icyc}.yaml \
                            >& ./qgLPF/output/exp_lpf/ens_var_cyc${icyc}_${anorbg}.log

   if [ $? -ne 0 ]; then
      echo "ERROR! Something went wrong running qg_ens_variance.x for cycle $icyc. Check ./qgLPF/output/exp_lpf/ens_var_cyc${icyc}_${anorbg}.log"
      break
   else
      echo "SUCCESS qg_ens_variance.x run for cycle $icyc $anorbg"
   fi
 done #anorbg loop
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done #icyc loop

ls -lhtr ./qgLPF/output/exp_lpf/da/*EnsVariance*nc

cd $JEDI_EDU
output_filename="/home/nonroot/shared-v2/EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt"
# Remove text file if it exists
[[ -f "$output_filename" ]] && rm -v $output_filename

firstdate="2009-12-31T00:00:00Z" 
curdate=$firstdate
for icyc in $(seq 1 $ncycles); do
   echo "Computing average ensemble variance for $curdate . . ."
   # Run for background mean file
   python ./plots_scripts/compute_ensvar_layers.py \
          $JEDI_EDU/qgLPF/output/exp_lpf/da/bgEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} bg"

   python ./plots_scripts/compute_ensvar_layers.py \
          $JEDI_EDU/qgLPF/output/exp_lpf/da/anEnsVariance.diag.${curdate}.nc \
          -v x -o $output_filename -l "cyc${icyc} an"
   
   # Advance curdate to the next day for the next cycle in the loop
   curdate=$(date -u -d "$curdate +1 day" +"%Y-%m-%dT%H:%M:%SZ")
done

cd $JEDI_EDU/plots_scripts

python plot_rmse_sawtooth_withvar.py -m 6e7 $JEDI_EDU/qgLPF/output/exp_lpf/plots/rmse_stream.txt \
                             $JEDI_EDU/qgLPF/output/exp_lpf/plots/ensvar_stream.txt \
                             -o $JEDI_EDU/qgLPF/output/exp_lpf/plots/rmse_sawtooth_withvar-40mem.png

| 20 members | 40 members |
|---|---|
| ![](images/rmse_sawtooth_withvar_lpf-20mem.png) | ![](images/rmse_sawtooth_withvar_lpf-40mem.png) |

The increased ensemble size has the expected result of reducing RMSEs and improving the spread-skill score of the filtering results.

# Summary

In this tutorial, we explored the LPF using the quasi-geostrophic model. We performed tests with a single observation and multiple observations to compare results to a similarly-configured LETKF, and investigated two key parameters that prevent weight collapse in the LPF: localization length scale and effective ensemble size.

### Key Concepts

1. **Delta-function approximation of PDFs:** Unlike the LETKF, the ensemble approximates a full distribution non-parametrically for the prior and posterior, rather than estimating only a mean and an error covariance.

2. **Localization:** Modulates the updates of members at distances away from observations, similar to covariance localization in the LETKF.

3. **Ensemble size:** Even with localization, the LPF requires larger ensemble sizes than ensemble Kalman filters to achieve satisfactory filtering results.
